# AI-Based Restoration of Degraded Images — Final Training Notebook

**Submitted model:** 12-block residual CNN with internal bicubic ×2 upsampling and Laplacian-aware reconstruction loss.

### Final validated setting

- Input: `128 × 128 × 1` degraded LR image
- Target: `256 × 256 × 1` ground-truth image
- Upsampling: bicubic ×2 **inside the model**
- Residual blocks: 12
- Final training loss: MSE + SSIM loss + Laplacian loss
- Data augmentation: none in the final model
- Validation PSNR: **27.7025 dB**
- Validation SSIM: **0.7481**

> Important: the available paired training data used for this submitted model is 128→256. The organizers' specification also mentions a 512→256 degradation path, but no native 512×512 GT was available in the supplied training data. This notebook therefore reproduces the actual 128→256 training experiment rather than fabricating a 512×512 ground truth.

## 1. Environment and imports

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)
print("Num GPUs:", len(tf.config.list_physical_devices("GPU")))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

## 2. Configuration

Update only the dataset paths when reproducing training.

The training data must contain paired files with the same six-digit ID, for example:

```text
GT/
    000000.npy
    000001.npy
    ...

LR/
    000000.npy
    000001.npy
    ...
```

The final model is trained on 128×128 LR → 256×256 GT pairs.

In [ ]:
# =========================
# CHANGE THESE PATHS
# =========================

GT_DIR = "/content/GT"
LR_DIR = "/content/LR"

MODEL_DIR = "/content/models"
os.makedirs(MODEL_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "final_laplacian_12block.keras"
)

# Training configuration
BATCH_SIZE = 32
EPOCHS = 25
VALIDATION_FRACTION = 0.20

# Loss weights used for the final model.
# Keep these equal to the values used for the submitted model.
SSIM_WEIGHT = 0.1
LAPLACIAN_WEIGHT = 0.1

## 3. Reproducible paired dataset split

We use explicit IDs rather than relying on dataset shuffling. This avoids the ID/pipeline mismatch that can occur when independently constructed datasets are shuffled.

The same ID is always used to locate the LR and GT pair.

In [ ]:
def get_ids(directory):
    ids = []

    for filename in os.listdir(directory):
        if filename.endswith(".npy"):
            stem = os.path.splitext(filename)[0]
            try:
                ids.append(f"{int(stem):06d}")
            except ValueError:
                pass

    return sorted(set(ids))


gt_ids = set(get_ids(GT_DIR))
lr_ids = set(get_ids(LR_DIR))

paired_ids = sorted(gt_ids.intersection(lr_ids))

print("GT files:", len(gt_ids))
print("LR files:", len(lr_ids))
print("Paired IDs:", len(paired_ids))

if len(paired_ids) == 0:
    raise RuntimeError("No matching LR/GT pairs were found.")

In [ ]:
# Deterministic train/validation split using random IDs.
rng = np.random.default_rng(SEED)

shuffled_ids = np.array(paired_ids, dtype=object)
rng.shuffle(shuffled_ids)

n_val = int(len(shuffled_ids) * VALIDATION_FRACTION)

val_ids = sorted(shuffled_ids[:n_val].tolist())
train_ids = sorted(shuffled_ids[n_val:].tolist())

print("Training pairs:", len(train_ids))
print("Validation pairs:", len(val_ids))

print("First training IDs:", train_ids[:5])
print("First validation IDs:", val_ids[:5])

## 4. Verify the paired data

This cell checks the expected shapes and confirms that the LR/GT files correspond to the same sample IDs.

In [ ]:
def load_pair(sample_id):
    lr = np.load(
        os.path.join(LR_DIR, f"{sample_id}.npy")
    ).astype(np.float32)

    gt = np.load(
        os.path.join(GT_DIR, f"{sample_id}.npy")
    ).astype(np.float32)

    return lr, gt


lr0, gt0 = load_pair(train_ids[0])

print("LR shape:", lr0.shape)
print("GT shape:", gt0.shape)

print("LR range:", lr0.min(), lr0.max())
print("GT range:", gt0.min(), gt0.max())

assert lr0.shape == (128, 128)
assert gt0.shape == (256, 256)

## 5. TensorFlow input pipeline

The final submitted model does **not** use random flips, Gaussian augmentation, or speckle augmentation. Those experiments were evaluated separately and did not improve the final validation result.

The raw LR values are passed to the model. In particular, LR values outside `[0, 1]` are not clipped before inference/training.

In [ ]:
def load_pair_tf(sample_id):
    sample_id = sample_id.numpy().decode("utf-8")

    lr, gt = load_pair(sample_id)

    lr = lr[..., None]
    gt = gt[..., None]

    return lr.astype(np.float32), gt.astype(np.float32)


def tf_load_pair(sample_id):
    lr, gt = tf.py_function(
        load_pair_tf,
        [sample_id],
        [tf.float32, tf.float32]
    )

    lr.set_shape((128, 128, 1))
    gt.set_shape((256, 256, 1))

    return lr, gt


def make_dataset(ids, training=False):
    ds = tf.data.Dataset.from_tensor_slices(
        np.asarray(ids, dtype=np.str_)
    )

    # No random shuffle is used in the final pipeline.
    ds = ds.map(
        tf_load_pair,
        num_parallel_calls=tf.data.AUTOTUNE
    )

    ds = ds.batch(BATCH_SIZE)

    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


train_ds = make_dataset(train_ids, training=True)
val_ds = make_dataset(val_ids, training=False)

for lr_batch, gt_batch in train_ds.take(1):
    print("LR batch:", lr_batch.shape)
    print("GT batch:", gt_batch.shape)

## 6. Residual block

In [ ]:
def residual_block(x, filters=64):
    shortcut = x

    y = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same",
        activation="relu"
    )(x)

    y = layers.Conv2D(
        filters,
        kernel_size=3,
        padding="same"
    )(y)

    y = layers.Add()([shortcut, y])
    y = layers.ReLU()(y)

    return y

## 7. Final 12-block residual model

The bicubic baseline is computed **inside the model**.

Therefore the training pipeline supplies raw 128×128 LR images. We do **not** bicubic-resize them before calling the model.

The model computes:

\[
	ext{output} = 	ext{bicubic}(LR) + 	ext{predicted residual}
\]

In [ ]:
def build_residual_model_12():
    inputs = keras.Input(
        shape=(128, 128, 1),
        name="lr"
    )

    # Bicubic ×2 baseline
    base = layers.Resizing(
        256,
        256,
        interpolation="bicubic",
        name="bicubic_resize"
    )(inputs)

    # Feature extraction
    x = layers.Conv2D(
        64,
        kernel_size=3,
        padding="same",
        activation="relu",
        name="feature_extraction"
    )(base)

    # 12 residual blocks
    for i in range(12):
        x = residual_block(
            x,
            filters=64
        )

    # Predict correction
    residual = layers.Conv2D(
        1,
        kernel_size=3,
        padding="same",
        name="predicted_residual"
    )(x)

    # Bicubic + learned residual
    output = layers.Add(
        name="residual_refinement"
    )([base, residual])

    return keras.Model(
        inputs=inputs,
        outputs=output,
        name="bicubic_residual_cnn_12block"
    )


model = build_residual_model_12()

model.summary()

## 8. Laplacian-aware loss

The final loss combines:

1. pixel-level MSE,
2. structural SSIM loss,
3. Laplacian loss.

The Laplacian term encourages the model to preserve local second-order structure and high-frequency changes without forcing the output residual to match the full noisy residual amplitude.

In [ ]:
def laplacian_response(x):
    kernel = tf.constant(
        [
            [0.0,  1.0, 0.0],
            [1.0, -4.0, 1.0],
            [0.0,  1.0, 0.0]
        ],
        dtype=tf.float32
    )

    kernel = tf.reshape(kernel, [3, 3, 1, 1])

    return tf.nn.conv2d(
        x,
        kernel,
        strides=[1, 1, 1, 1],
        padding="SAME"
    )


def laplacian_loss(y_true, y_pred):
    true_lap = laplacian_response(y_true)
    pred_lap = laplacian_response(y_pred)

    return tf.reduce_mean(
        tf.abs(true_lap - pred_lap)
    )


def final_loss(
    y_true,
    y_pred,
    ssim_weight=SSIM_WEIGHT,
    laplacian_weight=LAPLACIAN_WEIGHT
):
    mse = tf.reduce_mean(
        tf.square(y_true - y_pred)
    )

    ssim = tf.reduce_mean(
        tf.image.ssim(
            y_true,
            y_pred,
            max_val=1.0
        )
    )

    ssim_loss = 1.0 - ssim

    lap_loss = laplacian_loss(
        y_true,
        y_pred
    )

    return (
        mse
        + ssim_weight * ssim_loss
        + laplacian_weight * lap_loss
    )

## 9. Compile

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=final_loss,
    jit_compile=False
)

## 10. Callbacks

The best model is selected using validation loss. A separate checkpoint filename is used so that the final submitted model is not accidentally overwritten by another experiment.

In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH,
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    ),

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    )
]

## 11. Train the final model

In [ ]:
start_time = time.time()

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks
)

training_time = time.time() - start_time

print(
    f"Training time: {training_time:.2f} seconds"
)

## 12. Load the best checkpoint

In [ ]:
best_model = keras.models.load_model(
    BEST_MODEL_PATH,
    custom_objects={
        "final_loss": final_loss,
        "laplacian_loss": laplacian_loss,
        "laplacian_response": laplacian_response
    },
    compile=False
)

print("Loaded:", BEST_MODEL_PATH)
print("Input shape:", best_model.input_shape)
print("Output shape:", best_model.output_shape)

## 13. Validation metrics

Metrics are computed on the restored output against the 256×256 GT.

No bicubic image is supplied to the model externally.

In [ ]:
def evaluate_model(model, ids):
    psnr_values = []
    ssim_values = []

    for sample_id in ids:

        lr, gt = load_pair(sample_id)

        pred = model.predict(
            lr[None, ..., None],
            verbose=0
        )[0, ..., 0]

        psnr = tf.image.psnr(
            gt[..., None],
            pred[..., None],
            max_val=1.0
        ).numpy()

        ssim = tf.image.ssim(
            gt[..., None],
            pred[..., None],
            max_val=1.0
        ).numpy()

        psnr_values.append(float(psnr))
        ssim_values.append(float(ssim))

    return (
        float(np.mean(psnr_values)),
        float(np.mean(ssim_values)),
        np.asarray(psnr_values),
        np.asarray(ssim_values)
    )


val_psnr, val_ssim, val_psnr_all, val_ssim_all = evaluate_model(
    best_model,
    val_ids
)

print("===== FINAL VALIDATION =====")
print(f"Mean PSNR: {val_psnr:.4f} dB")
print(f"Mean SSIM: {val_ssim:.4f}")

## 14. Save validation metrics

This produces a small CSV that can be used when preparing the results slide.

In [ ]:
val_metrics_df = pd.DataFrame({
    "id": val_ids,
    "psnr": val_psnr_all,
    "ssim": val_ssim_all
})

val_metrics_path = "/content/final_validation_metrics.csv"

val_metrics_df.to_csv(
    val_metrics_path,
    index=False
)

print("Saved:", val_metrics_path)

## 15. Final model sanity check

The model is expected to produce 256×256 output from raw 128×128 LR.

The LR input is intentionally not clipped. Only the final output is clipped when preparing submission outputs.

In [ ]:
lr_test, gt_test = load_pair(val_ids[0])

pred_test = best_model.predict(
    lr_test[None, ..., None],
    verbose=0
)[0, ..., 0]

print("LR shape:", lr_test.shape)
print("GT shape:", gt_test.shape)
print("Prediction shape:", pred_test.shape)

print("LR range:", lr_test.min(), lr_test.max())
print("GT range:", gt_test.min(), gt_test.max())
print("Prediction range:", pred_test.min(), pred_test.max())

## 16. Visual validation

The comparison shows the degraded LR image, bicubic baseline, restored CNN output, and ground truth.

The bicubic image is created only for visualization; it is not fed into the model.

In [ ]:
import matplotlib.pyplot as plt

sample_id = val_ids[0]

lr, gt = load_pair(sample_id)

bicubic = tf.image.resize(
    lr[..., None],
    (256, 256),
    method="bicubic"
).numpy()[..., 0]

pred = best_model.predict(
    lr[None, ..., None],
    verbose=0
)[0, ..., 0]

fig, axes = plt.subplots(
    1, 4,
    figsize=(16, 4)
)

axes[0].imshow(lr, cmap="gray")
axes[0].set_title("LR 128×128")

axes[1].imshow(
    bicubic,
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[1].set_title("Bicubic 256×256")

axes[2].imshow(
    np.clip(pred, 0, 1),
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[2].set_title("12-block CNN")

axes[3].imshow(
    gt,
    cmap="gray",
    vmin=0,
    vmax=1
)
axes[3].set_title("GT")

for ax in axes:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 17. Export final model

The exported `.keras` file is the model placed in the submission `models/` directory.

For inference, it can be loaded with `compile=False`, so the custom training loss is not required by `run.py`.

In [ ]:
FINAL_EXPORT = os.path.join(
    MODEL_DIR,
    "final_laplacian_12block.keras"
)

best_model.save(FINAL_EXPORT)

print("Final model saved to:")
print(FINAL_EXPORT)

print("File size (MB):",
      os.path.getsize(FINAL_EXPORT) / (1024 ** 2))

## 18. Reproducibility information

Run the following in the final submission environment and save the complete output as the required environment specification.

In [ ]:
# In a terminal:
#
# pip freeze > requirements.txt
#
# The competition submission requires a complete pip freeze
# environment specification. Do not replace it with only the
# two packages used by the inference script.

# Final experiment record

The final model was selected after controlled experiments:

- 4 residual blocks + Laplacian loss: **27.5826 dB / 0.7438 SSIM**
- 8 residual blocks + Laplacian loss: **27.6599 dB / 0.7462 SSIM**
- 12 residual blocks + Laplacian loss: **27.7025 dB / 0.7481 SSIM**
- Residual scaling: rejected
- Speckle augmentation: rejected
- Gaussian augmentation: rejected
- Flip augmentation: rejected

### Final submitted training configuration

**12 residual blocks + MSE + SSIM + Laplacian loss, without augmentation.**

The final validation result was:

**PSNR = 27.7025 dB**

**SSIM = 0.7481**